# Semantic Segmentation with U-Net
## Google Colab에서 실행하는 Pet Segmentation 프로젝트

Author: Byunghyun Ban  
Date: 2020.07.24.

## 1. Google Drive 마운트 (선택사항)
데이터를 Google Drive에 저장한 경우 실행하세요

In [17]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 필요한 라이브러리 임포트

In [18]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import random
import os
from PIL import Image
import time
from matplotlib import pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.19.0


## 3. 데이터 업로드
data 폴더를 압축(zip)하여 업로드하거나, Google Drive에서 연결하세요

In [8]:
# # 방법 1: 파일 직접 업로드
# from google.colab import files
# import zipfile

# # data.zip 파일을 업로드하세요
# uploaded = files.upload()

# # 압축 해제
# for filename in uploaded.keys():
#     if filename.endswith('.zip'):
#         with zipfile.ZipFile(filename, 'r') as zip_ref:
#             zip_ref.extractall('.')
#         print(f'{filename} 압축 해제 완료')

In [19]:
# Google Drive 경로 확인
import os

# 데이터 경로 후보들
paths_to_check = [
    '/content/drive/MyDrive/Colab_Notebooks/data',
    '/content/drive/MyDrive/robot/data/data1',
    '/content/drive/MyDrive/robot/data/data',
    '/content/drive/MyDrive/robot/data',
]

print("📂 Google Drive 내 데이터 경로 탐색:\n")
for path in paths_to_check:
    if os.path.exists(path):
        print(f"✅ 발견: {path}")
        if os.path.exists(f"{path}/images"):
            img_count = len(os.listdir(f"{path}/images"))
            print(f"   └─ images 폴더: {img_count}개 파일")
        if os.path.exists(f"{path}/annotations"):
            ann_count = len(os.listdir(f"{path}/annotations"))
            print(f"   └─ annotations 폴더: {ann_count}개 파일")
        print()
    else:
        print(f"❌ 없음: {path}")

# MyDrive 전체 구조 확인
print("\n📁 MyDrive 최상위 폴더 목록:")
if os.path.exists('/content/drive/MyDrive'):
    items = os.listdir('/content/drive/MyDrive')
    for item in sorted(items)[:20]:  # 처음 20개만
        print(f"   - {item}")

📂 Google Drive 내 데이터 경로 탐색:

✅ 발견: /content/drive/MyDrive/Colab_Notebooks/data

❌ 없음: /content/drive/MyDrive/robot/data/data1
✅ 발견: /content/drive/MyDrive/robot/data/data

✅ 발견: /content/drive/MyDrive/robot/data


📁 MyDrive 최상위 폴더 목록:
   - .Trash-1000
   - Colab_Notebooks
   - Gemini
   - Manual_QuickReference
   - OpenCV Project Analysis V2.gslides
   - QuickSync
   - SD20-VV를 포장해서 해외로 보내는데 수신 측에서 포장을 해제한 후 부속품을 장착하는....gdoc
   - Smart Parking Guidance System v1.1 (1).gslides
   - Smart Parking Guidance System v1.1.gslides
   - Smart Parking Process.gslides
   - kakao_downloads
   - plc공부
   - robot
   - shkim
   - temp
   - temp_1767046710303.-1637123384.cpp
   - temp_1772333413822.778943166.xlsx
   - 교환폴더
   - 블로그 데이타
   - 삭제예정


## 4. U-Net 모델 정의

In [20]:
def build_unet(input_X, input_Y):
    """U-Net 모델을 생성합니다."""
    input_layer = keras.layers.Input((input_X, input_Y, 3))

    # 첫 번째 Convolution Block (Encoder)
    Conv1 = keras.layers.Conv2D(16, (3, 3), activation="relu", padding='same')(input_layer)
    Conv1 = keras.layers.Conv2D(16, (3, 3), activation="relu", padding='same')(Conv1)
    Pool1 = keras.layers.MaxPooling2D((2, 2))(Conv1)

    # 두 번째 Convolution Block
    Conv2 = keras.layers.Conv2D(32, (3, 3), activation="relu", padding='same')(Pool1)
    Conv2 = keras.layers.Conv2D(32, (3, 3), activation="relu", padding='same')(Conv2)
    Pool2 = keras.layers.MaxPooling2D((2, 2))(Conv2)

    # 세 번째 Convolution Block
    Conv3 = keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(Pool2)
    Conv3 = keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(Conv3)
    Pool3 = keras.layers.MaxPooling2D((2, 2))(Conv3)

    # 네 번째 Convolution Block
    Conv4 = keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(Pool3)
    Conv4 = keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(Conv4)
    Pool4 = keras.layers.MaxPooling2D((2, 2))(Conv4)

    # 다섯 번째 Convolution Block (Bottleneck)
    Conv5 = keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(Pool4)
    Conv5 = keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(Conv5)

    # 첫 번째 Upsampling Block (Decoder)
    Ups1 = keras.layers.Conv2DTranspose(128, (2, 2), strides=(2, 2))(Conv5)
    Ups1 = keras.layers.Concatenate()([Ups1, Conv4])
    Ups1_conv = keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same")(Ups1)
    Ups1_conv = keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same")(Ups1_conv)

    # 두 번째 Upsampling Block
    Ups2 = keras.layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding="same")(Ups1_conv)
    Ups2 = keras.layers.Concatenate()([Ups2, Conv3])
    Ups2_conv = keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(Ups2)
    Ups2_conv = keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(Ups2_conv)

    # 세 번째 Upsampling Block
    Ups3 = keras.layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding="same")(Ups2_conv)
    Ups3 = keras.layers.Concatenate()([Ups3, Conv2])
    Ups3_conv = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(Ups3)
    Ups3_conv = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(Ups3_conv)

    # 네 번째 Upsampling Block
    Ups4 = keras.layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding="same")(Ups3_conv)
    Ups4 = keras.layers.Concatenate()([Ups4, Conv1])
    Ups4_conv = keras.layers.Conv2D(16, (3, 3), activation="relu", padding="same")(Ups4)
    Ups4_conv = keras.layers.Conv2D(16, (3, 3), activation="relu", padding="same")(Ups4_conv)

    # Output Layer
    output_logit = keras.layers.Conv2D(1, (1, 1))(Ups4_conv)

    return keras.Model(inputs=input_layer, outputs=output_logit)

print("U-Net 모델 정의 완료!")

U-Net 모델 정의 완료!


## 5. 데이터 리더 클래스 정의

In [ ]:
import zipfile
from io import BytesIO
import glob

class DataReader():
    """Zip 파일에서 직접 데이터를 읽고 전처리하는 클래스"""
    def __init__(self):
        self.label = ["Background", "Pet"]

        self.train_X = []
        self.train_Y = []
        self.test_X = []
        self.test_Y = []

        self.read_data()

    def read_data(self):
        print("Reading Data from ZIP file...")

        # Google Colab 데이터 경로
        data_path = "/content/drive/MyDrive/Colab_Notebooks/data"

        print(f"📁 데이터 경로: {data_path}")
        print(f"📍 현재 작업 디렉토리: {os.getcwd()}\n")

        # Zip 파일 찾기
        zip_files = glob.glob(f"{data_path}/*.zip")

        if not zip_files:
            print(f"❌ 오류: '{data_path}' 폴더에서 zip 파일을 찾을 수 없습니다!")
            raise FileNotFoundError(f"No zip files found in {data_path}")

        zip_file_path = zip_files[0]
        print(f"✅ Zip 파일 발견: {os.path.basename(zip_file_path)}\n")

        # Zip 파일 열기
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Zip 내의 모든 파일 목록
            all_files = zip_ref.namelist()

            # images와 annotations 폴더의 파일들 찾기
            image_files = [f for f in all_files if '/images/' in f and f.endswith(('.jpg', '.png', '.JPG', '.PNG'))]
            annotation_files = [f for f in all_files if '/annotations/' in f and f.endswith(('.jpg', '.png', '.JPG', '.PNG'))]

            image_files.sort()
            annotation_files.sort()

            print(f"✅ 이미지 개수: {len(image_files)}")
            print(f"✅ 어노테이션 개수: {len(annotation_files)}\n")

            if len(image_files) == 0 or len(annotation_files) == 0:
                print(f"❌ 오류: 이미지 또는 어노테이션 파일을 찾을 수 없습니다!")
                print(f"    Zip 파일 구조: {all_files[:10]}")
                raise FileNotFoundError("Image or annotation files not found in zip")

            data = []

            print("🔄 데이터 로딩 중...")
            for i in range(len(image_files)):
                if (i + 1) % 100 == 0:
                    print(f"   진행: {i+1}/{len(image_files)} ({(i+1)/len(image_files)*100:.1f}%)")

                try:
                    # Zip 파일에서 이미지와 어노테이션 읽기
                    img_data = zip_ref.read(image_files[i])
                    ant_data = zip_ref.read(annotation_files[i])

                    # BytesIO를 사용해서 PIL Image로 변환
                    img = Image.open(BytesIO(img_data))
                    ant = Image.open(BytesIO(ant_data))

                    if img.mode != "RGB":
                        img = img.convert("RGB")

                    X = np.asarray(img) / 255.0

                    Y_temp = np.asarray(ant)[:, :, 0]
                    Y = np.zeros_like(Y_temp)
                    Y[Y_temp > 127.5] = 1.0

                    data.append((X, Y))
                    img.close()
                    ant.close()

                except Exception as e:
                    print(f"   ⚠️ 파일 처리 실패 ({i}): {e}")
                    continue

            print(f"   완료: {len(data)}/{len(image_files)} (100.0%)\n")

            random.shuffle(data)

            for i, el in enumerate(data):
                if i < 0.8 * len(data):
                    self.train_X.append(el[0])
                    self.train_Y.append(el[1])
                else:
                    self.test_X.append(el[0])
                    self.test_Y.append(el[1])

            self.train_X = np.asarray(self.train_X)
            self.train_Y = np.asarray(self.train_Y)
            self.test_X = np.asarray(self.test_X)
            self.test_Y = np.asarray(self.test_Y)

            # 데이터 읽기 완료
            print("=" * 50)
            print("✅ 데이터 로딩 완료!")
            print("=" * 50)
            print(f"Training X : {self.train_X.shape}")
            print(f"Training Y : {self.train_Y.shape}")
            print(f"Test X     : {self.test_X.shape}")
            print(f"Test Y     : {self.test_Y.shape}")
            print("=" * 50 + "\n")

    def show_processed_image(self, index=0):
        """전처리된 이미지를 시각화합니다."""
        plt.figure(figsize=(15, 8))
        image = self.train_X[index]
        annotation = self.train_Y[index]
        plt.subplot(1, 2, 1)
        plt.title("Original Image")
        plt.imshow(image)
        plt.subplot(1, 2, 2)
        plt.title("Annotation Mask")
        plt.imshow(annotation, cmap='gray')
        plt.show()

print("DataReader 클래스 정의 완료! (Zip 파일 직접 읽기 지원)")

DataReader 클래스 정의 완료!


## 6. 데이터 로드

In [22]:
# 데이터를 읽어옵니다.
dr = DataReader()

Reading Data...
📁 데이터 경로: /content/drive/MyDrive/Colab_Notebooks/data
📍 현재 작업 디렉토리: /content


⚠️ 오류: '/content/drive/MyDrive/Colab_Notebooks/data/images' 폴더를 찾을 수 없습니다!

해결 방법:
1. Google Drive가 마운트되었는지 확인하세요
2. Drive 경로에 data/data1 폴더가 있는지 확인하세요
3. 폴더 구조: /content/drive/MyDrive/Colab_Notebooks/data/images/ 및 /content/drive/MyDrive/Colab_Notebooks/data/annotations/



FileNotFoundError: /content/drive/MyDrive/Colab_Notebooks/data/images 폴더가 존재하지 않습니다.

## 7. 데이터 샘플 확인

In [ ]:
# 첫 번째 학습 이미지와 마스크를 확인합니다.
dr.show_processed_image(0)

## 8. 모델 생성 및 컴파일

In [ ]:
# U-Net 모델을 생성합니다.
model = build_unet(128, 128)

# 모델 구조 확인
model.summary()

In [ ]:
# 모델을 컴파일합니다.
loss = keras.losses.BinaryCrossentropy(from_logits=True)
model.compile(optimizer="adam", metrics=['accuracy'], loss=loss)

print("모델 컴파일 완료!")

## 9. 모델 학습

In [ ]:
# 학습 설정
EPOCHS = 50  # 에포크 수를 조정할 수 있습니다.

# 학습 시작
print("\n\n************ TRAINING START ************")
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(
    dr.train_X, dr.train_Y,
    epochs=EPOCHS,
    validation_data=(dr.test_X, dr.test_Y),
    callbacks=[early_stop]
)

print("\n\n************ TRAINING COMPLETE ************")

## 10. 학습 결과 시각화

In [ ]:
# Loss 그래프
train_history = history.history["loss"]
validation_history = history.history["val_loss"]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title("Loss History")
plt.xlabel("EPOCH")
plt.ylabel("LOSS Function")
plt.plot(train_history, "red", label="Train Loss")
plt.plot(validation_history, 'blue', label="Val Loss")
plt.legend()
plt.grid(True)

# Accuracy 그래프
train_acc = history.history["accuracy"]
validation_acc = history.history["val_accuracy"]

plt.subplot(1, 2, 2)
plt.title("Accuracy History")
plt.xlabel("EPOCH")
plt.ylabel("Accuracy")
plt.plot(train_acc, "red", label="Train Accuracy")
plt.plot(validation_acc, 'blue', label="Val Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 11. 예측 결과 시각화

In [ ]:
# 테스트 이미지에 대한 예측 수행
predictions = model.predict(dr.test_X)

# 여러 샘플의 예측 결과를 시각화합니다.
num_samples = min(5, len(dr.test_X))
fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))

for i in range(num_samples):
    # 원본 이미지
    axes[i, 0].imshow(dr.test_X[i])
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')

    # 실제 마스크
    axes[i, 1].imshow(dr.test_Y[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth Mask')
    axes[i, 1].axis('off')

    # 예측 마스크
    pred_mask = predictions[i, :, :, 0]
    pred_mask = (pred_mask > 0.5).astype(np.float32)
    axes[i, 2].imshow(pred_mask, cmap='gray')
    axes[i, 2].set_title('Predicted Mask')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 12. 결과 이미지 저장

In [ ]:
def save_segmentation_results(image, mask_y, model):
    """Segmentation 결과를 이미지로 저장합니다."""
    prediction = model.predict(image)[:, :, :, 0]
    prediction[prediction < 0] = 0
    pred_mask = (np.copy(image)*255).astype(np.uint8)[:, :, :, 0]
    pred_mask[prediction > 0.5] = 255

    mask = (np.copy(image)*255).astype(np.uint8)[:, :, :, 0]
    mask[mask_y > 0.5] = 255

    image_uint8 = (image * 255).astype(np.uint8)
    template = np.copy(image_uint8)[:, :, :, 0]

    mask = np.stack((template, mask, template), axis=3)
    pred_mask = np.stack((template, template, pred_mask), axis=3)

    if "results" not in os.listdir():
        os.mkdir("results")

    for i in range(len(image_uint8)):
        new_canvas = np.concatenate((image_uint8[i], mask[i], pred_mask[i]), axis=1)
        img = Image.fromarray(new_canvas)
        img.save("results/" + str(time.time()) + ".jpg")
        img.close()

    print("RESULT SAVED in 'results/' folder")

# 결과 저장
save_segmentation_results(dr.test_X, dr.test_Y, model)

## 13. 모델 저장

In [ ]:
# 모델을 저장합니다.
model.save("unet_segmentation_model.keras")
print("모델이 'unet_segmentation_model.keras'로 저장되었습니다.")

## 14. 결과 파일 다운로드 (선택사항)

In [ ]:
# 결과 폴더를 압축하여 다운로드
import shutil

# results 폴더를 zip으로 압축
if os.path.exists('results') and len(os.listdir('results')) > 0:
    shutil.make_archive('segmentation_results', 'zip', 'results')
    print("결과 파일이 'segmentation_results.zip'으로 압축되었습니다.")

    # Colab에서 다운로드
    files.download('segmentation_results.zip')
else:
    print("저장된 결과가 없습니다.")

## 15. 모델 다운로드 (선택사항)

In [ ]:
# 학습된 모델 다운로드
files.download('unet_segmentation_model.keras')